In [ ]:
import os
import shutil
import importlib
import copy
import glob
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import torch
import torchaudio

import phaselocknet_model
import util

importlib.reload(phaselocknet_model)
importlib.reload(util)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
device


In [ ]:
"""
Construct example torch model and load weights from checkpoint

* * * NOTE: This cell requires torch checkpoint to have been downloaded.
To convert tensorflow checkpoints to torch checkpoints, see cells below.
"""
dir_model = "models/sound_localization/simplified_IHC3000/arch01"
# dir_model = "models/spkr_word_recognition/simplified_IHC3000/arch0_0000"

# # Option 1: Manually construct the model from config files:
# with open(os.path.join(dir_model, "config.json"), "r") as f:
#     config_model = json.load(f)
# with open(os.path.join(dir_model, "arch.json"), "r") as f:
#     architecture = json.load(f)
# model = phaselocknet_model.Model(
#     config_model=config_model,
#     architecture=architecture,
#     input_shape=[2, 60000, 2],  # <-- [batch, timesteps @ 50 kHz sampling rate, channels==2] for sound_localization
#     config_random_slice={"size": [50, 10000], "buffer": [0, 500]},
# )

# Option 2: Use the `phaselocknet_model.get_model` function
model, config_model = phaselocknet_model.get_model(
    dir_model=dir_model,
    fn_config="config.json",
    fn_arch="arch.json",
)

# Load model weights from torch checkpoint
util.load_model_checkpoint(
    model=model.perceptual_model,
    dir_model=dir_model,
    fn_ckpt="ckpt_BEST.pt",
    weights_only=True,
)

model


In [ ]:
"""
Copy tensorflow model directories to new torch model directories for evaluation routine.
"""
# regex_dir_model_src = "../phaselocknet/models/sound_localization/simplified_IHC3000_delayed_integration/arch*"
regex_dir_model_src = "../phaselocknet/models/sound_localization/simplified_IHC3000/arch*"
# regex_dir_model_src = "../phaselocknet/models/spkr_word_recognition/simplified_IHC3000/arch*"
args_replace = ("../phaselocknet/", "../phaselocknet_torch/") # String replacement to map src to dst directory
for dir_model_src in glob.glob(regex_dir_model_src):
    # Build torch model object
    model, _ = phaselocknet_model.get_model(dir_model_src)
    # Load model weights from tensorflow checkpoint
    util.load_tf_model_checkpoint(
        model=model.perceptual_model,
        filename=os.path.join(dir_model_src, "ckpt_BEST"),
    )
    # Prepare destination directory for torch model
    assert args_replace[0] in dir_model_src
    dir_model_dst = dir_model_src.replace(*args_replace)
    if not os.path.exists(dir_model_dst):
        os.makedirs(dir_model_dst)
    # Save weights and configuration to torch model directory
    util.save_model_checkpoint(
        model=model.perceptual_model,
        dir_model=dir_model_dst,
        step=None,
        fn_ckpt="ckpt_BEST.pt",
    )
    # Copy `config.json` and `arch.json` to destination directory
    for basename in ["config.json", "arch.json"]:
        shutil.copyfile(
            os.path.join(dir_model_src, basename),
            os.path.join(dir_model_dst, basename),
        )
    print(f"[COMPLETE] {dir_model_dst=}\n")


In [ ]:
"""
Minimal example of how to load a `phaselocknet` model directory into a torch model.
Please be aware torch and tensorflow model outputs will not exactly match due to
the stochastic spike sampling and small differences in numerical precision.
"""

# # Sound localization network with simplified auditory nerve model (operates on audio)
# dir_model = "../phaselocknet/models/sound_localization/simplified_IHC3000_delayed_integration/arch01"

# # Sound localization network with detailed auditory nerve model (operates on pre-computed auditory nerve representations)
# dir_model = "../phaselocknet/models/sound_localization/IHC3000_delayed_integration/arch01"

# Word + voice recognition network with simplified auditory nerve model (operates on audio)
dir_model = "../phaselocknet/models/spkr_word_recognition/simplified_IHC3000/arch0_0000"

# # Word + voice recognition network with detailed auditory nerve model (operates on pre-computed auditory nerve representations)
# dir_model = "../phaselocknet/models/spkr_word_recognition/IHC3000/arch0_0000"

model, _ = phaselocknet_model.get_model(dir_model)

# Load model weights from tensorflow checkpoint
util.load_tf_model_checkpoint(
    model=model.perceptual_model,
    filename=os.path.join(dir_model, "ckpt_BEST"),
)
model.train(mode=False)
model.to(device)
assert not model.training


In [ ]:
regex_filenames = "../phaselocknet/stimuli/spkr_word_recognition/evaluation/pitch_altered_v00/stim*hdf5"

sr = 50000 if "sound_localization" in regex_filenames else 20000
num_steps_per_display = 10
dataset = util.HDF5Dataset(regex_filenames)


In [ ]:
example = dataset[10]

print("Example structure:")
for k, v in example.items():
    print("|__", k, v.shape if v.ndim > 0 else v, v.dtype)

x = torch.tensor(example["signal"])[None, ...]
sr_src = example["sr"]
resampler = torchaudio.transforms.Resample(
    orig_freq=sr_src,
    new_freq=sr,
)
print(f"[evaluate] resampling audio from {sr_src} to {sr} Hz")
if x.ndim > 2:
    x = torch.stack(
        [resampler(x[..., channel]) for channel in range(x.shape[-1])],
        axis=-1,
    )
else:
    x = resampler(x)

x = util.pad_or_trim_to_len(x, n=model.input_shape[1], dim=1)
out = model(x.to(device))

print("Model output:")
for k, v in out.items():
    print("|__", k, v.shape if v.ndim > 0 else v, v.dtype, np.argmax(v[0].cpu().detach().numpy()))
